# Hardware-aware NAS lab
Run top to bottom. Accuracy here is **synthetic**; latency comes from a device model (see README).

In [ ]:
import numpy as np
from hwnas import *
from hwnas.latency import mape, kendall_tau
space = SearchSpace()
print(space.n_layers, 'layers,', f'{space.size():,}', 'architectures')

## 1. Cost of two extreme architectures

In [ ]:
small = tuple(1 for _ in space.layers); big = tuple(5 for _ in space.layers)
for n, a in [('small', small), ('big', big)]:
    print(n, arch_metrics(space, a))

## 2. LUT vs end-to-end latency on each device

In [ ]:
rng = np.random.default_rng(0)
A = [space.random_arch(rng) for _ in range(300)]
for name, prof in PROFILES.items():
    lut = build_lut(space, prof)
    true = np.array([sim_latency(space, a, prof) for a in A]); est = np.array([lut.latency(a) for a in A])
    print(f'{name:9s} LUT MAPE {mape(est, true):6.2%}  tau {kendall_tau(est, true):.4f}')

## 3. Constrained search on the RPi-like device

In [ ]:
prof = PROFILES['rpi4']; lut = build_lut(space, prof); acc = SyntheticAccuracy(space)
cons = Constraints(max_latency=0.85e-3, max_sram=128*1024)
ev = Evaluator(space, acc, lut.latency)
best, hist = evolutionary_search(space, ev, cons, 500, np.random.default_rng(0))
print(space.describe(best.arch)); print(f'proxy acc {best.acc:.2f}, LUT {best.lat*1e3:.3f} ms, sim {sim_latency(space, best.arch, prof)*1e3:.3f} ms')

## 4. Pareto front

In [ ]:
ev = Evaluator(space, acc, lut.latency)
front, allc = nsga2(space, ev, Constraints(), 800, np.random.default_rng(0))
from hwnas.viz import plot_pareto
plot_pareto(allc, front, 'pareto.png'); print(len(front), 'points on the front -> pareto.png')

## Your turn
See `docs/LAB.md`. Start with Lab 1 question 3 and Lab 3 question 2 (cross-deployment).